# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

repo_url  = "https://github.com/min0712-cdl/HYU-AUE8088-PA2.git"
repo_name = "HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/AUE8088_PA2/outputs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "hybrid-rare-hard-joint"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# 1단계 — Level 3 best ViT-S 모델로 Set B 전체를 score.
level3_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "level3_best.pth"), map_location="cpu")
model = vit_small_patch16_224()
model.load_state_dict(level3_ckpt["state_dict"])
model = model.to(device).eval()

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성 (uncertainty) 계산: 1 - max-softmax 를 3개 head 평균.
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")
model = model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 2단계 — 희소 클래스, 희소 속성 조합, 불확실성, 오분류를 결합한 선별 전략.
import json
from collections import Counter

train_reference = BDDAttrDataset("../data/set_a", "train")
labels_b = {
    "weather": np.array([sample.weather for sample in set_b.samples]),
    "scene": np.array([sample.scene for sample in set_b.samples]),
    "timeofday": np.array([sample.timeofday for sample in set_b.samples]),
}

def minmax(values):
    values = np.asarray(values, dtype=np.float64)
    return (values - values.min()) / (values.max() - values.min() + 1e-12)

# Set A에서 적은 클래스일수록 높은 점수. 세 속성의 희소도를 평균.
attribute_rarity = np.zeros(len(set_b), dtype=np.float64)
for attr in ATTRIBUTES:
    counts = train_reference.class_counts(attr).numpy().astype(np.float64)
    per_sample = 1.0 / np.sqrt(np.maximum(counts[labels_b[attr]], 1.0))
    attribute_rarity += minmax(per_sample) / len(ATTRIBUTES)

# Weather×Scene×Time 조합이 Set A에서 드물수록 높은 점수.
joint_counts = Counter(
    (sample.weather, sample.scene, sample.timeofday) for sample in train_reference.samples
)
joint_rarity = minmax([
    1.0 / np.sqrt(joint_counts.get(
        (sample.weather, sample.scene, sample.timeofday), 0
    ) + 1.0)
    for sample in set_b.samples
])

# 현재 best 모델이 틀린 속성의 비율을 hard-example 신호로 사용.
error_rate = np.mean(np.stack([
    preds_b[attr] != labels_b[attr] for attr in ATTRIBUTES
], axis=1), axis=1)

uncertainty_score = minmax(uncertainty)
selection_score = (
    0.30 * attribute_rarity
    + 0.25 * joint_rarity
    + 0.25 * uncertainty_score
    + 0.20 * error_rate
)

K = 1000
selected_order = np.argsort(-selection_score)[:K]
rng = np.random.default_rng(SEED)
random_order = rng.choice(len(set_b), size=K, replace=False)

def records_from_indices(indices, reason):
    records = []
    for index in indices:
        sample = set_b.samples[int(index)]
        records.append({
            "image_id": sample.image_id,
            "weather": int(sample.weather),
            "scene": int(sample.scene),
            "timeofday": int(sample.timeofday),
            "selection_score": float(selection_score[index]),
            "uncertainty": float(uncertainty_score[index]),
            "attribute_rarity": float(attribute_rarity[index]),
            "joint_rarity": float(joint_rarity[index]),
            "error_rate": float(error_rate[index]),
            "reason": reason,
        })
    return records

picks = records_from_indices(selected_order, STRATEGY_NAME)
random_picks = records_from_indices(random_order, "random-baseline")
picks_path = os.path.join(OUTPUT_DIR, "level5_picks.json")
with open(picks_path, "w") as file:
    json.dump({
        "strategy": (
            "30% attribute rarity + 25% joint rarity + 25% uncertainty "
            "+ 20% attribute error rate"
        ),
        "num_picks": len(picks),
        "picks": picks,
    }, file, indent=2)
print(f"saved {len(picks)} picks to {picks_path}")

In [ ]:
# 3단계 — selected 250/500/1000 ablation과 random-1000 baseline을 동일 조건으로 학습.
val_ds = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
loader_val = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
FINETUNE_EPOCHS = 25
FINETUNE_LR = 1e-4

def train_with_picks(run_name, pick_records):
    extra = [
        (item["image_id"], item["weather"], item["scene"], item["timeofday"])
        for item in pick_records
    ]
    train_aug = BDDAttrDataset(
        "../data/set_a", "train", transform=train_transform(), extra_picks=extra
    )
    generator = torch.Generator(); generator.manual_seed(SEED)
    loader_tr = DataLoader(
        train_aug, batch_size=64, shuffle=True, num_workers=2,
        worker_init_fn=seed_worker, generator=generator, pin_memory=True,
    )

    set_seed(SEED, deterministic=True)
    trained_model = vit_small_patch16_224()
    trained_model.load_state_dict(level3_ckpt["state_dict"])
    trained_model = trained_model.to(device)
    optim = torch.optim.AdamW(
        trained_model.parameters(), lr=FINETUNE_LR, weight_decay=5e-2
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=FINETUNE_EPOCHS)
    losses = {attr: nn.CrossEntropyLoss() for attr in ATTRIBUTES}

    run_logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level5-{run_name}",
        config={
            "backbone": "vit_s16_level3_best", "strategy": run_name,
            "num_picks": len(pick_records), "epochs": FINETUNE_EPOCHS,
            "batch": 64, "lr": FINETUNE_LR, "seed": SEED,
        },
        tags=["level5", run_name],
    )
    trainer = MultiTaskTrainer(
        trained_model, optim, sched, losses, device,
        TrainConfig(epochs=FINETUNE_EPOCHS), logger=run_logger,
    )
    history = trainer.fit(loader_tr, loader_val)

    val_pred, _, val_tgt, _ = collect_predictions(trained_model, loader_val, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for attr in ATTRIBUTES:
        run_logger.log_confusion_matrix(f"final/cm_{attr}", cms[attr], CLASS_NAMES[attr])
        counts = Counter(item[attr] for item in pick_records)
        rows = [[CLASS_NAMES[attr][index], counts.get(index, 0)] for index in range(NUM_CLASSES[attr])]
        run_logger.log_table(f"picks/distribution_{attr}", ["class", "count"], rows)

    final_score = history["val_avg_mf1"][-1]
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"level5_{run_name}.pth")
    torch.save({
        "state_dict": trained_model.state_dict(), "history": history,
        "strategy": run_name, "num_picks": len(pick_records),
        "final_val_avg_mf1": final_score,
    }, checkpoint_path)
    run_logger.finish()

    trained_model = trained_model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "name": run_name, "num_picks": len(pick_records),
        "final_val_avg_mf1": final_score, "checkpoint": checkpoint_path,
    }

run_specs = [
    (f"{STRATEGY_NAME}-250", picks[:250]),
    (f"{STRATEGY_NAME}-500", picks[:500]),
    (f"{STRATEGY_NAME}-1000", picks),
    ("random-1000", random_picks),
]
results = [train_with_picks(name, records) for name, records in run_specs]

selected_1000 = next(result for result in results if result["name"].endswith("-1000") and result["name"] != "random-1000")
random_1000 = next(result for result in results if result["name"] == "random-1000")
di_score = (
    selected_1000["final_val_avg_mf1"] - random_1000["final_val_avg_mf1"]
) / max(random_1000["final_val_avg_mf1"], 1e-12)

summary_logger = WandbLogger(
    project=WANDB_PROJECT, run_name="level5-summary", tags=["level5", "summary"]
)
summary_logger.log_table(
    "level5/results", ["strategy", "num_picks", "final_avg_mf1"],
    [[result["name"], result["num_picks"], result["final_val_avg_mf1"]] for result in results],
)
summary_logger.log({"level5/di_score": di_score})
summary_logger.finish()

import shutil
final_checkpoint = os.path.join(CHECKPOINT_DIR, "level5_final.pth")
shutil.copy2(selected_1000["checkpoint"], final_checkpoint)
print(f"DI score = {di_score:.4f}")
print(f"Final checkpoint: {final_checkpoint}")

In [ ]:
# 4단계 — selected-1000 모델로 Kaggle 제출용 CSV 생성.
final_state = torch.load(os.path.join(CHECKPOINT_DIR, "level5_final.pth"), map_location="cpu")
model2 = vit_small_patch16_224()
model2.load_state_dict(final_state["state_dict"])
model2 = model2.to(device).eval()

test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
submission_path = os.path.join(OUTPUT_DIR, "level5_submission.csv")
write_submission(submission_path, ids_te, preds_te)
print(f"Kaggle submission 생성 완료: {submission_path}")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.